# Moving Object Detection with RPCA

This notebook implements a clean end-to-end pipeline:
1. Load video and convert to matrix `A` with shape `(pixels, frames)`.
2. Decompose `A = L + S` using IALM RPCA.
3. Reconstruct background/foreground videos.
4. Build cleaner binary foreground masks.


In [1]:
import numpy as np
import cv2 as cv
import time
import svd

## 1) Configuration and Data Loading

Flattening convention: each grayscale frame is reshaped in row-major order and stored as one column in `A`.


In [2]:
def video_to_matrix(
    path,
    resize_factor=0.5,
    frame_step=3,
    max_frames=300,
    dtype=np.float32,
    return_info=False,
):
    """Load video and convert to A with shape (pixels, frames)."""
    if resize_factor <= 0:
        raise ValueError("resize_factor must be > 0")
    if frame_step < 1:
        raise ValueError("frame_step must be >= 1")

    video = cv.VideoCapture(path)
    if not video.isOpened():
        raise FileNotFoundError(f"Cannot open video: {path}")

    src_fps = float(video.get(cv.CAP_PROP_FPS))
    if src_fps <= 0:
        src_fps = 25.0

    src_h = int(video.get(cv.CAP_PROP_FRAME_HEIGHT))
    src_w = int(video.get(cv.CAP_PROP_FRAME_WIDTH))
    src_frame_count = int(video.get(cv.CAP_PROP_FRAME_COUNT))

    frames = []
    frame_idx = 0
    while True:
        ok, frame = video.read()
        if not ok:
            break

        if frame_idx % frame_step == 0:
            gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
            if resize_factor != 1.0:
                gray = cv.resize(
                    gray,
                    None,
                    fx=resize_factor,
                    fy=resize_factor,
                    interpolation=cv.INTER_AREA,
                )
            frames.append(gray)
            if max_frames is not None and len(frames) >= max_frames:
                break

        frame_idx += 1

    video.release()

    if not frames:
        raise ValueError("No frames were read from video")

    frames = np.asarray(frames, dtype=dtype)
    frame_shape = frames.shape[1:]
    A = frames.reshape(frames.shape[0], -1).T
    effective_fps = src_fps / frame_step

    info = {
        "source_shape": (src_h, src_w),
        "source_frames": src_frame_count,
        "source_fps": src_fps,
        "resize_factor": resize_factor,
        "frame_step": frame_step,
        "used_frames": int(A.shape[1]),
        "matrix_shape": tuple(A.shape),
        "dtype": str(A.dtype),
        "effective_fps": effective_fps,
    }

    if return_info:
        return A, frame_shape, float(effective_fps), info
    return A, frame_shape, float(effective_fps)


VIDEO_PATH = "322.mp4"
RESIZE_FACTOR = 0.5
FRAME_STEP = 1
MAX_FRAMES = None
DTYPE = np.float32

A, frame_shape, fps, load_info = video_to_matrix(
    VIDEO_PATH,
    resize_factor=RESIZE_FACTOR,
    frame_step=FRAME_STEP,
    max_frames=MAX_FRAMES,
    dtype=DTYPE,
    return_info=True,
)

num_pixels, num_frames = A.shape
data_summary = {
    "A_shape": A.shape,
    "frame_shape": frame_shape,
    "fps": fps,
    "load_info": load_info,
}
data_summary


{'A_shape': (76800, 1501),
 'frame_shape': (240, 320),
 'fps': 25.0,
 'load_info': {'source_shape': (480, 640),
  'source_frames': 1501,
  'source_fps': 25.0,
  'resize_factor': 0.5,
  'frame_step': 1,
  'used_frames': 1501,
  'matrix_shape': (76800, 1501),
  'dtype': 'float32',
  'effective_fps': 25.0}}

## 2) IALM RPCA Solver

`solve_rpca` uses full SVD in each IALM iteration.


In [3]:
def estimate_spectral_norm(A, n_iter=8, seed=0):
    """Fast spectral norm estimate via power iteration."""
    A = np.asarray(A)
    m, n = A.shape
    if m == 0 or n == 0:
        return 0.0

    rng = np.random.default_rng(seed)
    v = rng.standard_normal(n).astype(A.dtype, copy=False)
    nv = np.linalg.norm(v)
    if nv == 0:
        return 0.0
    v /= nv

    for _ in range(n_iter):
        u = A @ v
        nu = np.linalg.norm(u)
        if nu == 0:
            return 0.0
        u /= nu

        v = A.T @ u
        nv = np.linalg.norm(v)
        if nv == 0:
            return 0.0
        v /= nv

    return float(np.linalg.norm(A @ v))


def solve_rpca(
    A,
    lam=None,
    mu=None,
    rho=1.3,
    max_iter=80,
    tol=1e-5,
    verbose=False,
):
    """Solve RPCA by IALM: A = L + S using full SVD."""
    A = np.asarray(A, dtype=np.float32)
    if A.ndim != 2:
        raise ValueError("A must be a 2D matrix")

    m, n = A.shape
    if m == 0 or n == 0:
        raise ValueError("A must be non-empty")

    if lam is None:
        lam = 1.0 / np.sqrt(max(m, n))

    norm_fro = np.linalg.norm(A, "fro")
    if norm_fro == 0:
        L = np.zeros_like(A)
        S = np.zeros_like(A)
        return L, S

    norm_two = estimate_spectral_norm(A, n_iter=8)
    if norm_two == 0:
        norm_two = norm_fro

    norm_inf = np.max(np.abs(A)) / lam
    dual_norm = max(norm_two, norm_inf)
    if dual_norm == 0:
        dual_norm = 1.0

    Y = A / dual_norm

    if mu is None:
        mu = 1.25 / (norm_two + 1e-8)
    mu_bar = mu * 1e5

    L = np.zeros_like(A)
    S = np.zeros_like(A)

    for k in range(1, max_iter + 1):
        temp = A - S + (Y / mu)
        tau = 1.0 / mu

        U, s, Vt = svd.svd(temp, 1e-15)

        s_thresh = np.maximum(s - tau, 0.0)
        rank = int(np.sum(s_thresh > 0))
        if rank > 0:
            L = (U[:, :rank] * s_thresh[:rank]) @ Vt[:rank, :]
        else:
            L.fill(0.0)

        temp = A - L + (Y / mu)
        S = np.sign(temp) * np.maximum(np.abs(temp) - (lam / mu), 0.0)

        residual = A - L - S
        err = np.linalg.norm(residual, "fro") / norm_fro

        if verbose and (k == 1 or k % 10 == 0 or err < tol):
            sparsity = float(np.mean(np.abs(S) > 1e-6))
            print(f"iter={k:4d} err={err:.3e} rank(L)={rank} sparsity(S)={sparsity:.4f}")

        if err < tol:
            break

        Y = Y + mu * residual
        mu = min(mu * rho, mu_bar)

    return L, S



## 3) Reconstruction and Mask Extraction


In [4]:
def matrix_to_frames(X, frame_shape):
    """Convert matrix (pixels, frames) back to frames (num_frames, H, W)."""
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError("X must be 2D with shape (pixels, num_frames)")

    h, w = frame_shape
    expected_pixels = h * w
    if X.shape[0] != expected_pixels:
        raise ValueError(
            f"Pixel count mismatch: X has {X.shape[0]} rows, expected {expected_pixels}"
        )

    return X.T.reshape(-1, h, w)


def save_grayscale_video(frames_u8, fps, path):
    """Save (num_frames, H, W) uint8 array as grayscale MP4."""
    if frames_u8.ndim != 3:
        raise ValueError("frames_u8 must have shape (num_frames, H, W)")
    if frames_u8.dtype != np.uint8:
        raise ValueError("frames_u8 must be uint8")

    n, h, w = frames_u8.shape
    fourcc = cv.VideoWriter_fourcc(*"mp4v")
    writer = cv.VideoWriter(path, fourcc, float(fps), (w, h), isColor=False)
    if not writer.isOpened():
        writer.release()
        raise RuntimeError(f"Failed to initialize video writer for: {path}")

    for i in range(n):
        writer.write(frames_u8[i])
    writer.release()
    return path


def save_rpca_videos(
    L,
    S,
    frame_shape,
    fps,
    background_path="background.mp4",
    foreground_path="foreground.mp4",
):
    """Save RPCA low-rank and sparse components as videos."""
    bg_frames = matrix_to_frames(L, frame_shape)
    fg_frames = matrix_to_frames(S, frame_shape)

    bg_u8 = np.clip(bg_frames, 0, 255).astype(np.uint8)

    fg_abs = np.abs(fg_frames)
    p99 = np.percentile(fg_abs, 99)
    scale = 255.0 / p99 if p99 > 1e-8 else 1.0
    fg_u8 = np.clip(fg_abs * scale, 0, 255).astype(np.uint8)

    save_grayscale_video(bg_u8, fps, background_path)
    save_grayscale_video(fg_u8, fps, foreground_path)
    return background_path, foreground_path


def filter_mask_components(mask, min_area=80, max_components=None):
    """Keep connected components by area and optional top-k selection."""
    if min_area <= 1 and max_components is None:
        return mask

    num_labels, labels, stats, _ = cv.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return mask

    kept = []
    for label in range(1, num_labels):
        area = int(stats[label, cv.CC_STAT_AREA])
        if area >= min_area:
            kept.append((area, label))

    if not kept:
        return np.zeros_like(mask)

    kept.sort(reverse=True)
    if max_components is not None:
        kept = kept[: int(max_components)]

    out = np.zeros_like(mask)
    for _, label in kept:
        out[labels == label] = 255
    return out


def extract_foreground_masks(
    S,
    frame_shape,
    threshold=20.0,
    threshold_mode="fixed",
    percentile=99.2,
    per_frame_percentile=True,
    open_kernel_size=3,
    close_kernel_size=7,
    min_threshold=2.0,
    core_percentile=99.7,
    core_dilate_size=9,
    min_component_area=80,
    max_components=3,
):
    """Extract cleaner binary masks from sparse RPCA component S."""
    sparse_frames = matrix_to_frames(S, frame_shape)
    abs_frames = np.abs(sparse_frames).astype(np.float32, copy=False)

    if threshold_mode == "fixed":
        used_threshold = float(threshold)
    elif threshold_mode == "percentile":
        used_threshold = float(np.percentile(abs_frames, percentile))
    else:
        raise ValueError("threshold_mode must be 'fixed' or 'percentile'")

    used_threshold = max(float(min_threshold), used_threshold)

    open_kernel = None
    if open_kernel_size and open_kernel_size > 1:
        open_kernel = cv.getStructuringElement(
            cv.MORPH_ELLIPSE, (int(open_kernel_size), int(open_kernel_size))
        )

    close_kernel = None
    if close_kernel_size and close_kernel_size > 1:
        close_kernel = cv.getStructuringElement(
            cv.MORPH_ELLIPSE, (int(close_kernel_size), int(close_kernel_size))
        )

    core_dilate_kernel = None
    if core_dilate_size and core_dilate_size > 1:
        core_dilate_kernel = cv.getStructuringElement(
            cv.MORPH_ELLIPSE, (int(core_dilate_size), int(core_dilate_size))
        )

    masks = np.zeros(abs_frames.shape, dtype=np.uint8)
    for i in range(abs_frames.shape[0]):
        frame_abs = abs_frames[i]

        frame_threshold = used_threshold
        if per_frame_percentile and threshold_mode == "percentile":
            frame_threshold = max(frame_threshold, float(np.percentile(frame_abs, percentile)))

        _, mask = cv.threshold(frame_abs, frame_threshold, 255, cv.THRESH_BINARY)
        mask = mask.astype(np.uint8, copy=False)

        if open_kernel is not None:
            mask = cv.morphologyEx(mask, cv.MORPH_OPEN, open_kernel)
        if close_kernel is not None:
            mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, close_kernel)

        if core_percentile is not None:
            core_thr = max(frame_threshold, float(np.percentile(frame_abs, core_percentile)))
            _, core = cv.threshold(frame_abs, core_thr, 255, cv.THRESH_BINARY)
            core = core.astype(np.uint8, copy=False)

            if np.any(core):
                if core_dilate_kernel is not None:
                    core = cv.dilate(core, core_dilate_kernel, iterations=1)
                mask = cv.bitwise_and(mask, core)

        mask = filter_mask_components(mask, min_area=min_component_area, max_components=max_components)
        masks[i] = mask

    return masks, used_threshold


def save_mask_video(mask_frames, fps, mask_path="foreground_mask.mp4"):
    """Save binary masks as grayscale MP4."""
    masks_u8 = np.asarray(mask_frames, dtype=np.uint8)
    return save_grayscale_video(masks_u8, fps, mask_path)


## 4) Run Pipeline


In [4]:
def run_pipeline(
    A,
    frame_shape,
    fps,
    rpca_kwargs=None,
    mask_kwargs=None,
    background_path="background.mp4",
    foreground_path="foreground.mp4",
    mask_path="foreground_mask.mp4",
):
    """Execute RPCA + export videos + export masks."""
    rpca_kwargs = dict(rpca_kwargs or {})
    mask_kwargs = dict(mask_kwargs or {})

    start = time.time()
    L, S = solve_rpca(A, **rpca_kwargs)
    elapsed = time.time() - start

    recon_err = np.linalg.norm(A - L - S, "fro") / np.linalg.norm(A, "fro")

    bg_path, fg_path = save_rpca_videos(
        L=L,
        S=S,
        frame_shape=frame_shape,
        fps=fps,
        background_path=background_path,
        foreground_path=foreground_path,
    )

    if "min_component_area" not in mask_kwargs:
        mask_kwargs["min_component_area"] = max(40, int(0.001 * frame_shape[0] * frame_shape[1]))

    mask_frames, used_threshold = extract_foreground_masks(
        S=S,
        frame_shape=frame_shape,
        **mask_kwargs,
    )
    saved_mask_path = save_mask_video(mask_frames, fps=fps, mask_path=mask_path)

    return {
        "reconstruction_error": float(recon_err),
        "estimated_rank_L": int(np.linalg.matrix_rank(L)),
        "sparsity_S": float(np.mean(np.abs(S) > 1e-6)),
        "rpca_elapsed_sec": float(elapsed),
        "mask_threshold_baseline": float(used_threshold),
        "background_path": bg_path,
        "foreground_path": fg_path,
        "mask_path": saved_mask_path,
    }


RUN_PIPELINE = True

RPCA_PARAMS = {
    "max_iter": 60,
    "tol": 1e-5,
    "rho": 1.3,
    "verbose": True,
}

MASK_PARAMS = {
    "threshold_mode": "percentile",
    "percentile": 99.2,
    "per_frame_percentile": True,
    "open_kernel_size": 3,
    "close_kernel_size": 7,
    "core_percentile": 99.7,
    "core_dilate_size": 9,
    "max_components": 3,
}

if RUN_PIPELINE:
    results = run_pipeline(
        A=A,
        frame_shape=frame_shape,
        fps=fps,
        rpca_kwargs=RPCA_PARAMS,
        mask_kwargs=MASK_PARAMS,
        background_path="background.mp4",
        foreground_path="foreground.mp4",
        mask_path="foreground_mask.mp4",
    )
    results
else:
    {
        "message": "Set RUN_PIPELINE=True to execute decomposition and export outputs.",
        "A_shape": A.shape,
        "frame_shape": frame_shape,
        "fps": fps,
    }


KeyboardInterrupt: 